In [4]:
import numpy as np
import pandas as pd
from pathlib import Path

protein_name = "YYCPETGTWY"
run_id       = 1
rmsd_center  = 0.10
delta        = 0.03
eps          = 1e-6

data_dir = Path(f"../data/{protein_name}/output")

def load_colvar(run_id):
    run_name  = f"run_{run_id:03}"
    colvar_fp = data_dir / run_name / f"HLDA_COLVAR_{run_id:03}"
    with colvar_fp.open() as f:
        header = next(line for line in f if line.startswith("#!"))
    cols = header.replace("#! FIELDS", "").split()
    return pd.read_csv(colvar_fp, sep=r"\s+", comment="#", names=cols)

def split_by_rmsd(df, center, half_width):
    mask_low  = (df["rmsd"] >= center - half_width) & (df["rmsd"] < center)
    mask_high = (df["rmsd"] >  center) & (df["rmsd"] <= center + half_width)
    return df[mask_low].copy(), df[mask_high].copy()

df = load_colvar(run_id)
folded_df, unfolded_df = split_by_rmsd(df, rmsd_center, delta)
print(f"{len(folded_df)} below, {len(unfolded_df)} above")

descriptor_cols = [c for c in df.columns if c.startswith("d")]
A = folded_df[descriptor_cols].to_numpy()
B = unfolded_df[descriptor_cols].to_numpy()

mu_A, mu_B = A.mean(0), B.mean(0)
Sigma_A = np.cov(A, rowvar=False)
Sigma_B = np.cov(B, rowvar=False)
Sigma_A += np.eye(Sigma_A.shape[0]) * eps
Sigma_B += np.eye(Sigma_B.shape[0]) * eps

inv_Sigma_A = np.linalg.inv(Sigma_A)
inv_Sigma_B = np.linalg.inv(Sigma_B)

W = (inv_Sigma_A + inv_Sigma_B) @ (mu_A - mu_B)
W /= np.linalg.norm(W)
W_final = W.copy()

print("\nHLDA weights:")
for name, w in zip(descriptor_cols, W_final):
    print(f"{name:>4}: {w:+.4f}")

s_folded   = (A @ W_final)[:10]
s_unfolded = (B @ W_final)[:10]
print("\ns_HLDA folded   :", np.round(s_folded, 4))
print("s_HLDA unfolded :", np.round(s_unfolded, 4))


622 below, 262 above

HLDA weights:
 d03: -0.1956
 d04: +0.1110
 d05: +0.2864
 d06: -0.0749
 d07: +0.0978
 d08: -0.0659
 d09: -0.0381
 d14: -0.2404
 d15: +0.1441
 d16: +0.0297
 d17: -0.2701
 d18: +0.2373
 d19: +0.0572
 d25: -0.2013
 d26: +0.1473
 d27: +0.1579
 d28: -0.2984
 d29: -0.0156
 d36: -0.1873
 d37: +0.0584
 d38: +0.4589
 d39: -0.0956
 d47: -0.1387
 d48: +0.1384
 d49: +0.0580
 d58: -0.3970
 d59: +0.0112
 d69: +0.0176

s_HLDA folded   : [0.0762 0.0486 0.0596 0.0472 0.0328 0.0512 0.0509 0.0413 0.0336 0.0636]
s_HLDA unfolded : [0.0455 0.0232 0.0314 0.0539 0.0731 0.0185 0.0239 0.015  0.0124 0.0079]
